# WaveRNN Training Notebook

This notebook trains the `WaveRNN` model defined in `wavernn.py` using the same manifest format used by the Tacotron pipeline.

Before running training, update the manifest paths in the config cell so they point to your `train_metadata.csv` and `test_metadata.csv` files. Keep `max_scaled_abs` aligned with the mel normalization used by the model that will provide conditioning mels at inference time.

In [ ]:
from __future__ import annotations

import json
import random
import sys
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import librosa
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import Audio, display
from torch import optim
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

ROOT = Path.cwd().resolve()
if not (ROOT / 'wavernn').exists():
    ROOT = ROOT.parent

WAVERNN_DIR = ROOT / 'wavernn'

sys.path.insert(0, str(WAVERNN_DIR))

from wavernn import WaveRNN


@dataclass
class TrainConfig:
    train_manifest: str = str(ROOT / 'data' / 'train_metadata.csv')
    val_manifest: str = str(ROOT / 'data' / 'test_metadata.csv')
    checkpoint_dir: str = str(WAVERNN_DIR / 'checkpoints')
    checkpoint_name: str = 'wavernn_last.pt'
    seed: int = 42

    batch_size: int = 8
    learning_rate: float = 1e-4
    weight_decay: float = 1e-6
    epochs: int = 20
    grad_clip: float = 4.0
    num_workers: int = 0
    eval_batches: int = 8

    sample_rate: int = 22050
    num_mels: int = 80
    n_fft: int = 1024
    window_size: int = 1024
    hop_length: int = 256
    fmin: int = 0
    fmax: int = 8000
    min_db: float = -100.0
    max_scaled_abs: float = 1.0

    upsample_scales: tuple[int, ...] = (4, 4, 16)
    n_classes: int = 256
    n_res_block: int = 10
    n_rnn: int = 512
    n_fc: int = 512
    kernel_size: int = 5
    n_hidden: int = 128
    n_output: int = 128
    segment_mel_frames: int = 64

    resume_from: str | None = None


cfg = TrainConfig()
cfg

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def load_wav(path_to_audio: str, sr: int = 22050) -> torch.Tensor:
    audio, _ = librosa.load(path_to_audio, sr=sr, mono=True)
    return torch.from_numpy(audio).float()


def amp_to_db(x: torch.Tensor, min_db: float = -100.0) -> torch.Tensor:
    clip_val = 10 ** (min_db / 20)
    return 20 * torch.log10(torch.clamp(x, min=clip_val))


def normalize(x: torch.Tensor, min_db: float = -100.0, max_abs_val: float = 4.0) -> torch.Tensor:
    x = (x - min_db) / -min_db
    x = 2 * max_abs_val * x - max_abs_val
    return torch.clamp(x, min=-max_abs_val, max=max_abs_val)


class AudioMelConversions:
    def __init__(
        self,
        num_mels: int = 80,
        sampling_rate: int = 22050,
        n_fft: int = 1024,
        window_size: int = 1024,
        hop_size: int = 256,
        fmin: int = 0,
        fmax: int = 8000,
        center: bool = False,
        min_db: float = -100.0,
        max_scaled_abs: float = 4.0,
    ) -> None:
        self.num_mels = num_mels
        self.sampling_rate = sampling_rate
        self.n_fft = n_fft
        self.window_size = window_size
        self.hop_size = hop_size
        self.fmin = fmin
        self.fmax = fmax
        self.center = center
        self.min_db = min_db
        self.max_scaled_abs = max_scaled_abs

        mel_basis = librosa.filters.mel(
            sr=self.sampling_rate,
            n_fft=self.n_fft,
            n_mels=self.num_mels,
            fmin=self.fmin,
            fmax=self.fmax,
        )
        self.spec2mel = torch.from_numpy(mel_basis).float()

    def audio2mel(self, audio: torch.Tensor, do_norm: bool = False) -> torch.Tensor:
        if not isinstance(audio, torch.Tensor):
            audio = torch.tensor(audio, dtype=torch.float32)

        window = torch.hann_window(self.window_size, device=audio.device)
        spectrogram = torch.stft(
            input=audio,
            n_fft=self.n_fft,
            hop_length=self.hop_size,
            win_length=self.window_size,
            window=window,
            center=self.center,
            pad_mode='reflect',
            normalized=False,
            onesided=True,
            return_complex=True,
        )

        spectrogram = torch.abs(spectrogram)
        mel = torch.matmul(self.spec2mel.to(spectrogram.device), spectrogram)
        mel = amp_to_db(mel, self.min_db)

        if do_norm:
            mel = normalize(mel, min_db=self.min_db, max_abs_val=self.max_scaled_abs)

        return mel


def quantize_waveform(waveform: torch.Tensor, n_classes: int) -> torch.Tensor:
    waveform = waveform.clamp(-1.0, 1.0)
    labels = torch.round((waveform + 1.0) * 0.5 * (n_classes - 1)).long()
    return labels.clamp_(0, n_classes - 1)


def labels_to_float(labels: torch.Tensor, n_classes: int) -> torch.Tensor:
    return 2.0 * labels.float() / (n_classes - 1) - 1.0


class WaveRNNDataset(Dataset):
    def __init__(self, manifest_path: str, config: TrainConfig) -> None:
        self.config = config
        self.metadata = pd.read_csv(manifest_path)
        self.audio_proc = AudioMelConversions(
            num_mels=config.num_mels,
            sampling_rate=config.sample_rate,
            n_fft=config.n_fft,
            window_size=config.window_size,
            hop_size=config.hop_length,
            fmin=config.fmin,
            fmax=config.fmax,
            center=False,
            min_db=config.min_db,
            max_scaled_abs=config.max_scaled_abs,
        )

        min_duration = ((config.segment_mel_frames - 1) * config.hop_length + config.window_size) / config.sample_rate
        if 'duration' in self.metadata.columns:
            self.metadata = self.metadata[self.metadata['duration'] >= min_duration].reset_index(drop=True)

        if len(self.metadata) == 0:
            raise ValueError('No samples are long enough for the configured segment_mel_frames value.')

        if config.segment_mel_frames < config.kernel_size:
            raise ValueError('segment_mel_frames must be >= kernel_size.')

        self.pad_frames = (config.kernel_size - 1) // 2
        self.target_samples = (config.segment_mel_frames - config.kernel_size + 1) * config.hop_length
        self.start_token = torch.tensor(config.n_classes // 2, dtype=torch.long)

    def __len__(self) -> int:
        return len(self.metadata)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        row = self.metadata.iloc[index]
        audio = load_wav(row['file_path'], sr=self.config.sample_rate).float()
        audio = audio.clamp(-1.0, 1.0)

        mel = self.audio_proc.audio2mel(audio, do_norm=True)
        total_mel_frames = mel.size(-1)
        if total_mel_frames < self.config.segment_mel_frames:
            raise ValueError(f"Sample at index {index} is too short after loading.")

        if total_mel_frames == self.config.segment_mel_frames:
            start_frame = 0
        else:
            start_frame = torch.randint(0, total_mel_frames - self.config.segment_mel_frames + 1, (1,)).item()

        mel = mel[:, start_frame : start_frame + self.config.segment_mel_frames]

        start_sample = start_frame * self.config.hop_length + self.pad_frames * self.config.hop_length
        end_sample = start_sample + self.target_samples
        if end_sample > audio.numel():
            audio = F.pad(audio, (0, end_sample - audio.numel()))

        target_labels = quantize_waveform(audio[start_sample:end_sample], self.config.n_classes)
        input_labels = torch.empty_like(target_labels)
        input_labels[0] = self.start_token
        input_labels[1:] = target_labels[:-1]

        input_waveform = labels_to_float(input_labels, self.config.n_classes).unsqueeze(0)
        return input_waveform, target_labels, mel.unsqueeze(0)


def collate_wavernn(batch):
    waveforms, targets, mels = zip(*batch)
    return torch.stack(waveforms), torch.stack(targets), torch.stack(mels)


set_seed(cfg.seed)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
Path(cfg.checkpoint_dir).mkdir(parents=True, exist_ok=True)

for manifest_path in (cfg.train_manifest, cfg.val_manifest):
    if not Path(manifest_path).is_file():
        raise FileNotFoundError(f'Manifest not found: {manifest_path}')

train_dataset = WaveRNNDataset(cfg.train_manifest, cfg)
val_dataset = WaveRNNDataset(cfg.val_manifest, cfg)

train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=device.type == 'cuda',
    collate_fn=collate_wavernn,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=device.type == 'cuda',
    collate_fn=collate_wavernn,
)

model = WaveRNN(
    upsample_scales=list(cfg.upsample_scales),
    n_classes=cfg.n_classes,
    hop_length=cfg.hop_length,
    n_res_block=cfg.n_res_block,
    n_rnn=cfg.n_rnn,
    n_fc=cfg.n_fc,
    kernel_size=cfg.kernel_size,
    n_freq=cfg.num_mels,
    n_hidden=cfg.n_hidden,
    n_output=cfg.n_output,
).to(device)

optimizer = optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)

num_params = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
print(json.dumps(asdict(cfg), indent=2))
print(f'Using device: {device}')
print(f'Train samples: {len(train_dataset)} | Val samples: {len(val_dataset)}')
print(f'Trainable parameters: {num_params:,}')

In [ ]:
def save_checkpoint(model, optimizer, epoch: int, global_step: int, config: TrainConfig, filename: str) -> Path:
    checkpoint_path = Path(config.checkpoint_dir) / filename
    torch.save(
        {
            'epoch': epoch,
            'global_step': global_step,
            'config': asdict(config),
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        },
        checkpoint_path,
    )
    return checkpoint_path


def maybe_resume(model, optimizer, config: TrainConfig) -> tuple[int, int]:
    if config.resume_from is None:
        return 0, 0

    checkpoint = torch.load(config.resume_from, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    print(f"Resumed from {config.resume_from}")
    return checkpoint['epoch'] + 1, checkpoint['global_step']


def compute_loss(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    logits = logits.squeeze(1).reshape(-1, cfg.n_classes)
    targets = targets.reshape(-1)
    return F.cross_entropy(logits, targets)


@torch.no_grad()
def evaluate(model, dataloader, max_batches: int | None = None) -> float:
    model.eval()
    losses = []

    for batch_index, (waveform_inputs, targets, mels) in enumerate(dataloader):
        waveform_inputs = waveform_inputs.to(device)
        targets = targets.to(device)
        mels = mels.to(device)

        logits = model(waveform_inputs, mels)
        losses.append(compute_loss(logits, targets).item())

        if max_batches is not None and batch_index + 1 >= max_batches:
            break

    model.train()
    return sum(losses) / max(len(losses), 1)


@torch.no_grad()
def preview_generation(model, dataloader, config: TrainConfig):
    model.eval()
    _, targets, mels = next(iter(dataloader))
    mels = mels.to(device)
    generated, _ = model.infer(mels[0, 0].unsqueeze(0))

    target_audio = labels_to_float(targets[0], config.n_classes).cpu()
    generated_audio = generated[0, 0].detach().cpu()

    print('Target audio preview')
    display(Audio(target_audio.numpy(), rate=config.sample_rate))
    print('Generated audio preview')
    display(Audio(generated_audio.numpy(), rate=config.sample_rate))
    model.train()


def train_model(model, optimizer, train_loader, val_loader, config: TrainConfig):
    start_epoch, global_step = maybe_resume(model, optimizer, config)
    history = {'train_loss': [], 'val_loss': []}

    for epoch in range(start_epoch, config.epochs):
        model.train()
        running_loss = 0.0
        start_time = time.time()

        progress = tqdm(train_loader, desc=f'Epoch {epoch + 1}/{config.epochs}', leave=False)
        for step_in_epoch, (waveform_inputs, targets, mels) in enumerate(progress, start=1):
            waveform_inputs = waveform_inputs.to(device)
            targets = targets.to(device)
            mels = mels.to(device)

            optimizer.zero_grad(set_to_none=True)
            logits = model(waveform_inputs, mels)
            loss = compute_loss(logits, targets)
            loss.backward()

            if config.grad_clip is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)

            optimizer.step()

            running_loss += loss.item()
            global_step += 1
            progress.set_postfix(loss=f'{running_loss / step_in_epoch:.4f}')

        train_loss = running_loss / max(len(train_loader), 1)
        val_loss = evaluate(model, val_loader, max_batches=config.eval_batches)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)

        elapsed = time.time() - start_time
        print(
            f'Epoch {epoch + 1}/{config.epochs} | '
            f'train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | '
            f'time={elapsed:.1f}s | global_step={global_step}'
        )

        checkpoint_path = save_checkpoint(model, optimizer, epoch, global_step, config, config.checkpoint_name)
        print(f'Saved checkpoint to {checkpoint_path}')

    return history

In [ ]:
history = train_model(model, optimizer, train_loader, val_loader, cfg)
history

In [ ]:
preview_generation(model, val_loader, cfg)